In [1]:
import os
import sys
import pickle as pkl
import pandas as pd
import numpy as np
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm

/home/cyrus/venvs/uts-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Import Dataset
Dataset contains SBERT question, human response, agent response, and agent summary vectors. 

In [2]:
sys.path.append(str(Path.cwd().parent))

ROOT    = Path(os.getcwd()).resolve().parents[0]
DATA    = ROOT / "data"
WEIGHTS = ROOT / "weights"

with open(DATA / "concat_train_set.pkl", "rb") as file:
    train_set = pkl.load(file)

with open(DATA / "concat_test_set.pkl", "rb") as file:
    test_set = pkl.load(file)

train_df    = pd.DataFrame(train_set)
test_df     = pd.DataFrame(test_set)

print("Train set:", train_df.shape)
print("Test set:", train_df.shape)

Train set: (1708, 12)
Test set: (1708, 12)


## Load BERT Models

In [3]:
import torch
from dataclasses import dataclass, field
from transformers import (
    BertForQuestionAnswering, 
    AutoTokenizer,
)
from sentence_transformers import SentenceTransformer

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [5]:
bert_model = BertForQuestionAnswering.from_pretrained("deepset/bert-base-cased-squad2")
bert_model.to(device)
bert_tokenizer = AutoTokenizer.from_pretrained("deepset/bert-base-cased-squad2")
sentence_model = SentenceTransformer('paraphrase-MiniLM-L6-v2').to(device)

# Count BERT's total number of trainable parameters by summing the size of every weight tensor.
bert_num_params = sum(p.numel() for p in bert_model.parameters())
# Print the count, so "small model" isn't just a label in a markdown cell -- see the real number.
print(f"BERT ({bert_model.config.name_or_path}) parameters: {bert_num_params:,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7763.40it/s]
[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/bert-base-cased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11598.61it/s]


BERT (deepset/bert-base-cased-squad2) parameters: 107,721,218


In [6]:
@dataclass
class Config:
    # Training config
    epochs: int         = 50

    # Data shape
    question_dim: int   = sentence_model.get_sentence_embedding_dimension()
    bert_dim: int       = sentence_model.get_sentence_embedding_dimension()

    # Latent sizes
    style_dim: int      = sentence_model.get_sentence_embedding_dimension()
    content_dim: int    = sentence_model.get_sentence_embedding_dimension()
    context_dim: int    = sentence_model.get_sentence_embedding_dimension()

    disc_dim: int       = sentence_model.get_sentence_embedding_dimension()
    gene_dim: int       = sentence_model.get_sentence_embedding_dimension()

    # Optimisation
    lr_generator: float     = 1e-4
    lr_discriminator: float = 1e-4
    batch_size: int         = 32

cfg = Config(epochs=50)

/tmp/ipykernel_155325/691886234.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  question_dim: int   = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_155325/691886234.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  bert_dim: int       = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_155325/691886234.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  style_dim: int      = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_155325/691886234.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  content_dim: int    = sentence_model.get_sentence_embedding_dimension()
/tmp/ipykernel_155325/691886234.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed t

In [7]:
alt_state = torch.load(WEIGHTS / "alt_state.pt", weights_only=False, map_location="cpu")
mix_state = torch.load(WEIGHTS / "mix_state.pt", weights_only=False, map_location="cpu")

alt_config = alt_state["config"]
mix_config = mix_state["config"]

alt_model = alt_state["model"]
mix_model = mix_state["model"]

alt_model.to(device)
mix_model.to(device)

SBERTDisentangler(
  (style_encoder): StyleEncoder(
    (mlp): Sequential(
      (0): Linear(in_features=384, out_features=384, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=384, out_features=384, bias=True)
    )
  )
  (content_encoder): ContentEncoder(
    (mlp): Sequential(
      (0): Linear(in_features=384, out_features=384, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=384, out_features=384, bias=True)
    )
  )
  (context_encoder): ContextEncoder(
    (mlp): Sequential(
      (0): Linear(in_features=384, out_features=384, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=384, out_features=384, bias=True)
    )
  )
  (generator): SBERTGenerator(
    (mlp): Sequential(
      (0): Linear(in_features=1152, out_features=384, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=384, out_features=384, bias=True)
    )
  )
  (discriminator): SBERTDiscriminator(
    (mlp): Sequ

## Probe Training Configuration

In [9]:
@torch.no_grad()
def compute_baselines(model, loader, device):
    model.eval()

    all_questions = []
    all_human_style = []
    all_agent_style = []
    all_human_content = []
    all_agent_content = []
    all_human_context = []
    all_agent_context = []

    for batch in loader:
        question = batch["question"].to(device)
        human = batch["human"].to(device)
        agent = batch["agent"].to(device)

        # Encode the paired human and AI responses
        style_h, content_h, context_h = model.encode(human)
        style_a, content_a, context_a = model.encode(agent)

        all_questions.append(question.cpu())
        all_human_style.append(style_h.cpu())
        all_agent_style.append(style_a.cpu())
        all_human_content.append(content_h.cpu())
        all_agent_content.append(content_a.cpu())
        all_human_context.append(context_h.cpu())
        all_agent_context.append(context_a.cpu())

    # Concatenate all examples across batches
    human_style = torch.cat(all_human_style, dim=0)
    agent_style = torch.cat(all_agent_style, dim=0)
    human_content = torch.cat(all_human_content, dim=0)
    agent_content = torch.cat(all_agent_content, dim=0)
    human_context = torch.cat(all_human_context, dim=0)
    agent_context = torch.cat(all_agent_context, dim=0)

    # Pool human and agent targets for each representation.
    # Each response receives equal weight.
    all_content = torch.cat([human_content, agent_content], dim=0)
    all_context = torch.cat([human_context, agent_context], dim=0)

    # Calculate the mean target vectors
    baselines = {
        "content": all_content.mean(dim=0),
        "context": all_context.mean(dim=0),
    }

    return baselines

In [10]:
from train.pairs import *

train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)
train_human     = torch.tensor(train_df["hr_bert"].to_list(), dtype=torch.float32)
train_agent     = torch.tensor(train_df["as_bert"].to_list(), dtype=torch.float32)

test_question   = torch.tensor(test_df["question_bert"].to_list(), dtype=torch.float32)
test_human      = torch.tensor(test_df["hr_bert"].to_list(), dtype=torch.float32)
test_agent      = torch.tensor(test_df["as_bert"].to_list(), dtype=torch.float32)

train_dataset = SummaryPairDataset(
    train_question,
    train_human,
    train_agent
)
test_dataset = SummaryPairDataset(
    test_question,
    test_human,
    test_agent
)

train_loader    = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
test_loader     = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=True) 

/tmp/ipykernel_155325/3805931428.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)


In [11]:
@dataclass
class ProbeConfig:
    epochs: int     = 50
    batch_size: int = 32
    lr: float       = 1e-4

    mix_style_base      = 0.5
    mix_content_base    = compute_baselines(mix_model, train_loader, device)["content"]
    mix_context_base    = compute_baselines(mix_model, train_loader, device)["context"]
    
    alt_style_base      = 0.5
    alt_content_base    = compute_baselines(alt_model, train_loader, device)["content"]
    alt_context_base    = compute_baselines(alt_model, train_loader, device)["context"]

In [12]:
import torch.nn as nn

class StyleContentProbe(nn.Module):
    def __init__(self, model):
        super().__init__()

        cfg = model.cfg
        self.head = nn.Linear(cfg.style_dim, cfg.content_dim)

    def forward(self, x):
        return self.head(x)

class StyleContextProbe(nn.Module):
    def __init__(self, model):
        super().__init__()

        cfg = model.cfg
        self.head = nn.Linear(cfg.style_dim, cfg.context_dim)

    def forward(self, x):
        return self.head(x)

class ContentStyleProbe(nn.Module):
    def __init__(self, model):
        super().__init__()

        cfg = model.cfg
        self.head = nn.Linear(cfg.content_dim, 1)

    def forward(self, x):
        return self.head(x)

class ContentContextProbe(nn.Module):
    def __init__(self, model):
        super().__init__()

        cfg = model.cfg
        self.head = nn.Linear(cfg.content_dim, cfg.context_dim)

    def forward(self, x):
        return self.head(x)

class ContextStyleProbe(nn.Module):
    def __init__(self, model):
        super().__init__()

        cfg = model.cfg
        self.head = nn.Linear(cfg.context_dim, 1)

    def forward(self, x):
        return self.head(x)

class ContextContentProbe(nn.Module):
    def __init__(self, model):
        super().__init__()

        cfg = model.cfg
        self.head = nn.Linear(cfg.context_dim, cfg.content_dim)

    def forward(self, x):
        return self.head(x)

In [13]:
from train.pairs import *

train_question  = torch.tensor(train_df["question_bert"].to_list(), dtype=torch.float32)
train_human     = torch.tensor(train_df["hr_bert"].to_list(), dtype=torch.float32)
train_agent     = torch.tensor(train_df["as_bert"].to_list(), dtype=torch.float32)

test_question   = torch.tensor(test_df["question_bert"].to_list(), dtype=torch.float32)
test_human      = torch.tensor(test_df["hr_bert"].to_list(), dtype=torch.float32)
test_agent      = torch.tensor(test_df["as_bert"].to_list(), dtype=torch.float32)

train_dataset = SummaryPairDataset(
    train_question,
    train_human,
    train_agent
)
test_dataset = SummaryPairDataset(
    test_question,
    test_human,
    test_agent
)

train_loader    = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
test_loader     = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=True) 

In [14]:
from itertools import chain

# Define probe config
probe_cfg = ProbeConfig()

# Define probes
mix_probes = {
    "mix_style_content"   : StyleContentProbe(mix_model).to(device),
    "mix_style_context"   : StyleContextProbe(mix_model).to(device),
    "mix_content_style"   : ContentStyleProbe(mix_model).to(device),
    "mix_content_context" : ContentContextProbe(mix_model).to(device),
    "mix_context_style"   : ContextStyleProbe(mix_model).to(device),
    "mix_context_content" : ContextContentProbe(mix_model).to(device)
}

alt_probes = {
    "alt_style_content"   : StyleContentProbe(alt_model).to(device),
    "alt_style_context"   : StyleContextProbe(alt_model).to(device),
    "alt_content_style"   : ContentStyleProbe(alt_model).to(device),
    "alt_content_context" : ContentContextProbe(alt_model).to(device),
    "alt_context_style"   : ContextStyleProbe(alt_model).to(device),
    "alt_context_content" : ContextContentProbe(alt_model).to(device)
}

# Define optimizers
mix_probe_optim = torch.optim.AdamW(
    chain.from_iterable(value.parameters() for key, value in mix_probes.items()),
    lr=probe_cfg.lr
)

alt_probe_optim = torch.optim.AdamW(
    chain.from_iterable(value.parameters() for key, value in alt_probes.items()),
    lr=probe_cfg.lr
)

In [15]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

probe_specs = {
    "style_content": ("style", "content"),
    "style_context": ("style", "context"),
    "content_style": ("content", "style"),
    "content_context": ("content", "context"),
    "context_style": ("context", "style"),
    "context_content": ("context", "content"),
}

model_configs = [
    (mix_model, mix_probes, mix_probe_optim, "mix"),
    (alt_model, alt_probes, alt_probe_optim, "alt"),
]

# Freeze encoders.
for model, _, _, _ in model_configs:
    model.eval()
    for param in model.parameters():
        param.requires_grad_(False)

In [16]:
for epoch in range(probe_cfg.epochs):
    # Track each probe's loss separately across batches.
    epoch_losses = {
        model_name: {probe_type: [] for probe_type in probe_specs}
        for _, _, _, model_name in model_configs
    }

    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}"):
        batch = {k: v.to(device) for k, v in batch.items()}

        # Extract frozen representations.
        with torch.no_grad():
            representations = {}

            for model, _, _, model_name in model_configs:
                h_s, h_c, h_x = model.encode(batch["human"])
                a_s, a_c, a_x = model.encode(batch["agent"])

                representations[model_name] = {
                    "style": {"human": h_s, "agent": a_s},
                    "content": {"human": h_c, "agent": a_c},
                    "context": {"human": h_x, "agent": a_x},
                }

        for _, probes, optimizer, model_name in model_configs:
            vecs = representations[model_name]
            optimizer.zero_grad(set_to_none=True)

            batch_probe_losses = {}

            for name, probe in probes.items():
                probe_type = name.removeprefix(f"{model_name}_")
                source, target = probe_specs[probe_type]

                response_losses = []

                for response in ("human", "agent"):
                    x = vecs[source][response]
                    pred = probe(x)

                    if target == "style":
                        label = float(response == "agent")
                        y = torch.full_like(pred, label)

                        loss = F.binary_cross_entropy_with_logits(pred, y)
                    else:
                        y = vecs[target][response].detach()

                        loss = F.mse_loss(pred, y)

                    response_losses.append(loss)

                # Average human and agent losses for this probe.
                batch_probe_losses[probe_type] = torch.stack(response_losses).mean()

            # Equal weighting of all six probe losses.
            total_loss = torch.stack(list(batch_probe_losses.values())).mean()

            total_loss.backward()
            optimizer.step()

            # Record individual losses.
            for probe_type, loss in batch_probe_losses.items():
                epoch_losses[model_name][probe_type].append(loss.item())

    # Print mean loss for every probe.
    print(f"\nEpoch {epoch + 1}/{probe_cfg.epochs}")

    for model_name in ("mix", "alt"):
        print(f"\n{model_name.upper()} MODEL")

        for probe_type, losses in epoch_losses[model_name].items():
            mean_loss = sum(losses) / max(len(losses), 1)

            print(f"  {probe_type:20s}: {mean_loss:.6f}")

Epoch 1: 100%|██████████| 54/54 [00:00<00:00, 88.41it/s] 



Epoch 1/50

MIX MODEL
  style_content       : 0.015321
  style_context       : 0.021513
  content_style       : 0.690957
  content_context     : 0.021495
  context_style       : 0.692250
  context_content     : 0.017112

ALT MODEL
  style_content       : 0.022943
  style_context       : 0.027868
  content_style       : 0.691064
  content_context     : 0.029727
  context_style       : 0.695163
  context_content     : 0.024259


Epoch 2: 100%|██████████| 54/54 [00:00<00:00, 179.24it/s]



Epoch 2/50

MIX MODEL
  style_content       : 0.011920
  style_context       : 0.017321
  content_style       : 0.686460
  content_context     : 0.015927
  context_style       : 0.690424
  context_content     : 0.011541

ALT MODEL
  style_content       : 0.018096
  style_context       : 0.021049
  content_style       : 0.685179
  content_context     : 0.018942
  context_style       : 0.691605
  context_content     : 0.014358


Epoch 3: 100%|██████████| 54/54 [00:00<00:00, 128.79it/s]


...


Epoch 48/50

MIX MODEL
  style_content       : 0.001521
  style_context       : 0.003276
  content_style       : 0.566725
  content_context     : 0.002855
  context_style       : 0.659069
  context_content     : 0.000680

ALT MODEL
  style_content       : 0.002262
  style_context       : 0.003311
  content_style       : 0.555304
  content_context     : 0.002337
  context_style       : 0.651409
  context_content     : 0.001348


Epoch 49: 100%|██████████| 54/54 [00:00<00:00, 200.27it/s]



Epoch 49/50

MIX MODEL
  style_content       : 0.001498
  style_context       : 0.003228
  content_style       : 0.565140
  content_context     : 0.002822
  context_style       : 0.658515
  context_content     : 0.000674

ALT MODEL
  style_content       : 0.002224
  style_context       : 0.003256
  content_style       : 0.553776
  content_context     : 0.002306
  context_style       : 0.651003
  context_content     : 0.001337


Epoch 50: 100%|██████████| 54/54 [00:00<00:00, 203.76it/s]


Epoch 50/50

MIX MODEL
  style_content       : 0.001474
  style_context       : 0.003179
  content_style       : 0.563842
  content_context     : 0.002787
  context_style       : 0.658350
  context_content     : 0.000669

ALT MODEL
  style_content       : 0.002190
  style_context       : 0.003207
  content_style       : 0.552812
  content_context     : 0.002284
  context_style       : 0.650639
  context_content     : 0.001325


## Evaluate Probes

In [17]:
model_configs = [
    (mix_model, mix_probes, "mix"),
    (alt_model, alt_probes, "alt"),
]

# Evaluation mode for encoders and probes.
for model, probes, _ in model_configs:
    model.eval()
    for probe in probes.values():
        probe.eval()

baselines = {
    "mix": {
        "content": probe_cfg.mix_content_base,
        "context": probe_cfg.mix_context_base,
    },
    "alt": {
        "content": probe_cfg.alt_content_base,
        "context": probe_cfg.alt_context_base,
    },
}

results = {}

In [18]:
with torch.no_grad():
    for model, probes, model_name in model_configs:

        # Baselines calculated from this model's TRAINING set.
        model_baselines = baselines[model_name]

        metrics = {
            name: {
                "correct": 0,
                "count": 0,
                "cosine_sum": 0.0,
                "mse_sum": 0.0,
                "baseline_cosine_sum": 0.0,
                "baseline_mse_sum": 0.0,
            }
            for name in probe_specs
        }

        for batch in tqdm(test_loader, desc=f"Testing {model_name}"):
            batch = {k: v.to(device) for k, v in batch.items()}

            h_s, h_c, h_x = model.encode(batch["human"])
            a_s, a_c, a_x = model.encode(batch["agent"])

            vecs = {
                "style": {"human": h_s, "agent": a_s},
                "content": {"human": h_c, "agent": a_c},
                "context": {"human": h_x, "agent": a_x},
            }

            for name, probe in probes.items():
                probe_type = name.removeprefix(f"{model_name}_")
                source, target = probe_specs[probe_type]
                m = metrics[probe_type]

                for response in ("human", "agent"):
                    source_vec = vecs[source][response]
                    target_vec = vecs[target][response]

                    pred = probe(source_vec)

                    if target == "style":
                        # Human = 0, agent = 1.
                        label = int(response == "agent")
                        predicted_label = (pred >= 0).long().view(-1)

                        m["correct"] += (
                            predicted_label == label
                        ).sum().item()
                        m["count"] += predicted_label.numel()

                    else:
                        # Trained probe metrics.
                        m["cosine_sum"] += (
                            F.cosine_similarity(
                                pred, target_vec, dim=-1
                            ).sum().item()
                        )

                        m["mse_sum"] += (
                            (pred - target_vec)
                            .pow(2)
                            .mean(dim=-1)
                            .sum()
                            .item()
                        )

                        # Mean-target baseline: same prediction
                        # for every example, regardless of source.
                        baseline_vec = model_baselines[target].to(
                            device=target_vec.device,
                            dtype=target_vec.dtype,
                        )

                        baseline_pred = baseline_vec.unsqueeze(0).expand_as(
                            target_vec
                        )

                        m["baseline_cosine_sum"] += (
                            F.cosine_similarity(
                                baseline_pred, target_vec, dim=-1
                            ).sum().item()
                        )

                        m["baseline_mse_sum"] += (
                            (baseline_pred - target_vec)
                            .pow(2)
                            .mean(dim=-1)
                            .sum()
                            .item()
                        )

                        m["count"] += target_vec.size(0)

        # Aggregate metrics.
        results[model_name] = {}

        print(f"\n{'=' * 75}")
        print(f"{model_name.upper()} MODEL — TEST RESULTS")
        print(f"{'=' * 75}")

        for probe_type, m in metrics.items():
            n = max(m["count"], 1)
            results[model_name][probe_type] = {}

            if probe_specs[probe_type][1] == "style":
                accuracy = m["correct"] / n
                results[model_name][probe_type]["accuracy"] = accuracy

                print(
                    f"{probe_type:20s} | "
                    f"Accuracy: {accuracy:.4f} | "
                    f"Chance: 0.5000"
                )

            else:
                cosine = m["cosine_sum"] / n
                mse = m["mse_sum"] / n
                baseline_cosine = m["baseline_cosine_sum"] / n
                baseline_mse = m["baseline_mse_sum"] / n

                results[model_name][probe_type].update({
                    "cosine": cosine,
                    "mse": mse,
                    "baseline_cosine": baseline_cosine,
                    "baseline_mse": baseline_mse,
                    "cosine_gain": cosine - baseline_cosine,
                    "mse_reduction": baseline_mse - mse,
                })

                print(
                    f"{probe_type:20s} | "
                    f"Probe: Cos={cosine:.4f}, MSE={mse:.6f} | "
                    f"Baseline: Cos={baseline_cosine:.4f}, "
                    f"MSE={baseline_mse:.6f} | "
                    f"ΔCos={cosine - baseline_cosine:+.4f}, "
                    f"ΔMSE={baseline_mse - mse:+.6f}"
                )

Testing mix: 100%|██████████| 6/6 [00:00<00:00, 87.53it/s]



MIX MODEL — TEST RESULTS
style_content        | Probe: Cos=0.9371, MSE=0.001459 | Baseline: Cos=0.1177, MSE=0.013125 | ΔCos=+0.8194, ΔMSE=+0.011666
style_context        | Probe: Cos=0.9129, MSE=0.003153 | Baseline: Cos=0.2850, MSE=0.018680 | ΔCos=+0.6278, ΔMSE=+0.015527
content_style        | Accuracy: 0.7237 | Chance: 0.5000
content_context      | Probe: Cos=0.9210, MSE=0.002824 | Baseline: Cos=0.2850, MSE=0.018680 | ΔCos=+0.6360, ΔMSE=+0.015856
context_style        | Accuracy: 0.6395 | Chance: 0.5000
context_content      | Probe: Cos=0.9697, MSE=0.000715 | Baseline: Cos=0.1177, MSE=0.013125 | ΔCos=+0.8520, ΔMSE=+0.012409


Testing alt: 100%|██████████| 6/6 [00:00<00:00, 316.08it/s]


ALT MODEL — TEST RESULTS
style_content        | Probe: Cos=0.9462, MSE=0.002079 | Baseline: Cos=0.2451, MSE=0.019779 | ΔCos=+0.7011, ΔMSE=+0.017700
style_context        | Probe: Cos=0.9367, MSE=0.003195 | Baseline: Cos=0.4934, MSE=0.022843 | ΔCos=+0.4432, ΔMSE=+0.019648
content_style        | Accuracy: 0.7368 | Chance: 0.5000
content_context      | Probe: Cos=0.9556, MSE=0.002256 | Baseline: Cos=0.4934, MSE=0.022843 | ΔCos=+0.4621, ΔMSE=+0.020587
context_style        | Accuracy: 0.6211 | Chance: 0.5000
context_content      | Probe: Cos=0.9662, MSE=0.001319 | Baseline: Cos=0.2451, MSE=0.019779 | ΔCos=+0.7211, ΔMSE=+0.018459
